In [0]:
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
dbutils.widgets.text("schema_prefix", "retail", "Schema Prefix")
dbutils.widgets.dropdown("fail_on_critical", "true", ["true", "false"], "Fail Pipeline on Critical Error")

In [0]:
catalog = dbutils.widgets.get("catalog_name")
schema_prefix = dbutils.widgets.get("schema_prefix")
fail_on_critical = dbutils.widgets.get("fail_on_critical") == "true"

silver_db = f"{catalog}.{schema_prefix}_silver"
gold_db = f"{catalog}.{schema_prefix}_gold"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE {gold_db}")

print("Silver:", silver_db)
print("Gold:", gold_db)

Silver: workspace.retail_silver
Gold: workspace.retail_gold


In [0]:
from pyspark.sql.functions import col
import time

def log_result(table_name, check_name, status, failure_count, is_critical):
    log_data = [(table_name, check_name, status, int(failure_count), is_critical, time.strftime("%Y-%m-%d %H:%M:%S"))]
    log_df = (spark.createDataFrame(log_data, schema=["table_name","check_name","status","failure_count","is_critical","run_timestamp"])
              .withColumn("run_timestamp", col("run_timestamp").cast("timestamp")))
    target = f"{gold_db}.data_quality_logs"
    if not spark.catalog.tableExists(target):
        log_df.write.format("delta").mode("overwrite").saveAsTable(target)
    else:
        log_df.write.format("delta").mode("append").saveAsTable(target)

def check_nulls(table_fq, columns, is_critical=True):
    df = spark.table(table_fq)
    all_passed = True
    for c in columns:
        n = df.filter(col(c).isNull()).count()
        status = "FAIL" if n > 0 else "PASS"
        print(f"{status}: {table_fq}.{c} — {n} nulls")
        log_result(table_fq, f"null_check_{c}", status, n, is_critical)
        if n > 0: all_passed = False
    return all_passed

def check_uniqueness(table_fq, key_columns, is_critical=True):
    df = spark.table(table_fq)
    dup = df.count() - df.select(key_columns).distinct().count()
    status = "FAIL" if dup > 0 else "PASS"
    print(f"{status}: {table_fq} — {dup} duplicate keys")
    log_result(table_fq, "uniqueness_check_" + "_".join(key_columns), status, dup, is_critical)
    return dup == 0

def check_range(table_fq, col_name, min_val, max_val, is_critical=True):
    df = spark.table(table_fq)
    bad = df.filter(col(col_name).isNotNull() & ((col(col_name) < min_val) | (col(col_name) > max_val))).count()
    status = "FAIL" if bad > 0 else "PASS"
    print(f"{status}: {table_fq}.{col_name} — {bad} out of [{min_val},{max_val}]")
    log_result(table_fq, f"range_check_{col_name}", status, bad, is_critical)
    return bad == 0

def check_referential_integrity(fact_table, fact_fk, dim_table, dim_pk, is_critical=True):
    orphans = (spark.table(fact_table).alias("f")
               .join(spark.table(dim_table).alias("d"), col(f"f.{fact_fk}") == col(f"d.{dim_pk}"), "left_anti")
               .filter(col(f"f.{fact_fk}").isNotNull()).count())
    status = "FAIL" if orphans > 0 else "PASS"
    print(f"{status}: {fact_table}.{fact_fk} — {orphans} orphaned rows")
    log_result(fact_table, f"fk_check_{fact_fk}", status, orphans, is_critical)
    return orphans == 0

In [0]:
critical_failures = 0

if not check_nulls(f"{silver_db}.customers", ["customer_id", "customer_unique_id"]): critical_failures += 1
if not check_nulls(f"{silver_db}.orders", ["order_id", "customer_id"]): critical_failures += 1
if not check_uniqueness(f"{silver_db}.customers", ["customer_id"]): critical_failures += 1
if not check_uniqueness(f"{silver_db}.orders", ["order_id"]): critical_failures += 1

if not check_nulls(f"{gold_db}.FactSales", ["sales_key", "order_id", "customer_key", "product_key"]): critical_failures += 1
if not check_uniqueness(f"{gold_db}.FactSales", ["sales_key"]): critical_failures += 1
if not check_range(f"{gold_db}.FactSales", "price", 0.0, 100000.0): critical_failures += 1
check_range(f"{gold_db}.FactSales", "review_score", 1, 5, is_critical=False)

if not check_referential_integrity(f"{gold_db}.FactSales", "customer_key", f"{gold_db}.DimCustomer", "customer_key"): critical_failures += 1
if not check_referential_integrity(f"{gold_db}.FactSales", "product_key", f"{gold_db}.DimProduct", "product_key"): critical_failures += 1
if not check_referential_integrity(f"{gold_db}.FactSales", "seller_key", f"{gold_db}.DimSeller", "seller_key"): critical_failures += 1

print(f"\nTotal critical failures: {critical_failures}")
if critical_failures > 0 and fail_on_critical:
    raise Exception(f"{critical_failures} critical checks failed — see {gold_db}.data_quality_logs")

PASS: workspace.retail_silver.customers.customer_id — 0 nulls
PASS: workspace.retail_silver.customers.customer_unique_id — 0 nulls
PASS: workspace.retail_silver.orders.order_id — 0 nulls
PASS: workspace.retail_silver.orders.customer_id — 0 nulls
PASS: workspace.retail_silver.customers — 0 duplicate keys
PASS: workspace.retail_silver.orders — 0 duplicate keys
PASS: workspace.retail_gold.FactSales.sales_key — 0 nulls
PASS: workspace.retail_gold.FactSales.order_id — 0 nulls
PASS: workspace.retail_gold.FactSales.customer_key — 0 nulls
PASS: workspace.retail_gold.FactSales.product_key — 0 nulls
PASS: workspace.retail_gold.FactSales — 0 duplicate keys
PASS: workspace.retail_gold.FactSales.price — 0 out of [0.0,100000.0]
PASS: workspace.retail_gold.FactSales.review_score — 0 out of [1,5]
PASS: workspace.retail_gold.FactSales.customer_key — 0 orphaned rows
PASS: workspace.retail_gold.FactSales.product_key — 0 orphaned rows
PASS: workspace.retail_gold.FactSales.seller_key — 0 orphaned rows

Tot